# Preprocess the Raw Dataset for Linear Models

Linear models (like Logistic Regression or SVM) assume that the input features have a linear relationship with the log-odds of the target. To perform well, they require:

- **Numerical Encoding**: Converting text to numbers.
- **Scaling**: Ensuring all features have the same range (e.g., 0 to 1) so that features with large magnitudes don't dominate the model weights.
- **Handling Multicollinearity**: Careful selection of encoding methods.

### 1. Environment Setup

Add the project root to `sys.path` to allow for modular imports of local configurations and ingestion scripts.

In [ ]:
import sys
from pathlib import Path

# Get the path to 'irrigation-prediction-system' (the project root)
root_path: Path = Path.cwd().parent.parent.parent
sys.path.append(str(root_path))

### 2. Import Required Libraries and Custom Configs

In [ ]:
import numpy as np
import pandas as pd
from prefect.settings import PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW, temporary_settings
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder, OrdinalEncoder

from src.configs.data_cfg import FALSE_VALUES, NA_VALUES, OPTIMIZED_DTYPES, TRUE_VALUES
from src.configs.settings import Settings, get_settings
from src.pipeline.ingestion import load_dataset

# Setup configs
settings: Settings = get_settings()

# Setup data paths
raw_data_path: Path = settings.RAW_DATA_DIR / settings.RAW_DATA_FILENAME
preprocessed_filepath: Path = settings.EXPERIMENTS_DATA_DIR / settings.LINEAR_FILENAME

### 3. Load the Dataset

In [ ]:
try:
    with temporary_settings(updates={PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW: "ignore"}):
        df: pd.DataFrame = load_dataset(
            raw_data_path,
            sep=",",
            header=0,
            index_col="id",
            true_values=TRUE_VALUES,
            false_values=FALSE_VALUES,
            dtype=OPTIMIZED_DTYPES,
            na_values=NA_VALUES,
            keep_default_na=True,
            on_bad_lines="warn",
            float_precision="round_trip",
            skipinitialspace=True,
            encoding="utf-8",
            encoding_errors="replace",
            memory_map=True,
            low_memory=False,
        )
except Exception as e:
    sys.stderr.write(f"Ingestion failed: {e}\n")

### 4. Split the Columns into Groups

In [ ]:
# Target
target_col: str = "Irrigation_Need"

# Feature groups
bool_cols: tuple[str, ...] = ("Mulching_Used",)
ordinal_cols: tuple[str, ...] = ("Crop_Growth_Stage",)
nominal_cols: tuple[str, ...] = (
    "Soil_Type",
    "Crop_Type",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
)

# Ordinal category orders
crop_growth_stage_categories: tuple[str, ...] = ("Sowing", "Vegetative", "Flowering", "Harvest")
irrigation_need_categories: tuple[str, ...] = ("Low", "Medium", "High")

# Numeric and categorica columns
numeric_cols: list[str] = df.select_dtypes(["Float32"]).columns.tolist()
categorical_cols: list[str] = df.select_dtypes(["category", "boolean"]).columns.tolist()
categorical_cols.remove(target_col)

### 5. Split the Dataset into Train-Test

In [ ]:
# Separate features (X) and target (y)
X: pd.DataFrame = df.drop(columns=[target_col])
y: pd.Series = df[target_col]

# Encode the target before split
encoder: OrdinalEncoder = OrdinalEncoder(categories=[list(irrigation_need_categories)])
y_encoded = encoder.fit_transform(y.to_numpy().reshape(-1, 1))
y = pd.Series(y_encoded.flatten(), index=y.index, name=target_col, dtype=np.int8)

# Stratified train-test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 7. Create Function Transformers for Custom Encoding

In [ ]:
def _make_bool_transformer() -> FunctionTransformer:
    """
    Create custom transformer for boolean-like columns.

    Returns
    -------
    FunctionTransformer
        Transformer converting input to {0,1}.
    """

    def transform_bool(X: pd.DataFrame) -> pd.DataFrame:
        """Convert boolean-like values to int8."""
        return X.astype(bool).astype(np.int8)

    return FunctionTransformer(transform_bool, validate=False, feature_names_out="one-to-one")


def _make_ordinal_transformer() -> OrdinalEncoder:
    """
    Create ordinal encoder with predefined category ordering.

    Returns
    -------
    OrdinalEncoder
        Encoder with explicit category mapping.
    """
    mapping: dict[str, list[str]] = {"Crop_Growth_Stage": list(crop_growth_stage_categories)}
    categories: list[list[str]] = [mapping[col] for col in ordinal_cols]
    return OrdinalEncoder(
        categories=categories, handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.int8
    )


def _make_nominal_transformer() -> OneHotEncoder:
    """
    Create OneHotEncoder for nominal categorical features.

    Returns
    -------
    OneHotEncoder
        Encoder with:
        - dense output
        - unknown category handling
        - int8 dtype
    """
    return OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype=np.int8)

### 8. Build the Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("bool", _make_bool_transformer(), list(bool_cols)),
        ("ordinal", _make_ordinal_transformer(), list(ordinal_cols)),
        ("nominal", _make_nominal_transformer(), list(nominal_cols)),
        ("numeric", MinMaxScaler(), list(numeric_cols)),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

pipeline: Pipeline = Pipeline(steps=[("preprocessor", preprocessor)], verbose=True)
pipeline.set_output(transform="pandas")

### 9. Encode the Datasets and Save Locally

In [ ]:
pipeline.fit(X_train)
X_train_enc: pd.DataFrame = pipeline.transform(X_train)
X_test_enc: pd.DataFrame = pipeline.transform(X_test)
X_train_enc.info()

In [ ]:
X_train_enc.head(5)

### 10. Save the datasets as parquet

In [ ]:
# Stack the train and test sets vertically
pd.concat(
    [pd.concat([X_train_enc, y_train], axis=1), pd.concat([X_test_enc, y_test], axis=1)], axis=0
).to_parquet(preprocessed_filepath)

print(f"Preprocessed dataset saved at {settings.EXPERIMENTS_DATA_DIR}")